[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/058.%20The%20ABC_XYZ%20Inventory%20Classification%20Problem/P58-Tier-3_executed.ipynb)

---


# 58. The ABC/XYZ Inventory Classification Problem

## Tier 3: Hybrid Genetic Algorithm and Simulated Annealing

### Purpose and scope

This notebook uses a hybrid genetic algorithm with simulated annealing to search for ABC/XYZ classification thresholds and objective weights. The inputs are SKU annual values and monthly demand observations. The outputs are a threshold configuration, the resulting nine-category classification, proxy performance metrics, and search-history visualizations.

The method is a stochastic metaheuristic: it explores candidate configurations but does not prove that the returned configuration is a global optimum. Reproducible random seeds make the example repeatable in a compatible software environment; they do not turn the heuristic into an exact optimization method.

### Model assumptions

- Annual value is the measure used for ABC classification. Larger values are assigned to class A, intermediate values to B, and smaller values to C.
- Demand variability is measured by the sample coefficient of variation, \(CV=s/\bar{x}\), using the monthly observations for each SKU. Smaller CV values correspond to X, then Y, and larger values to Z.
- The performance measures are illustrative proxies, not estimates from observed inventory costs, fill rates, or historical classification changes. In particular, the stability proxy measures within-category similarity in CV; it does not measure temporal classification stability.
- Cost and service proxies use fixed category-level factors. The resulting scores are useful for demonstrating how a weighted objective guides the search, but should not be interpreted as validated business forecasts.
- The example excludes lead times, ordering and holding costs, stockout penalties, supplier constraints, and demand forecasts.

### Method at a glance

A chromosome stores four thresholds and three nonnegative objective weights. The genetic algorithm initializes a population, selects parents by tournament selection, applies uniform crossover and Gaussian mutation, and retains an elite candidate. Simulated annealing periodically refines a strong candidate by accepting improving moves and sometimes accepting worsening moves. This probabilistic acceptance can help the search move away from a local optimum.

### ABC/XYZ classification

Let \(v_i\) be the annual value of SKU \(i\), and let \(CV_i\) be its demand coefficient of variation. For thresholds \(V_A>V_B\) and \(CV_X<CV_Y\), the classes are

\[
\operatorname{ABC}(i)=
\begin{cases}
A, & v_i\geq V_A,\\
B, & V_B\leq v_i<V_A,\\
C, & v_i<V_B,
\end{cases}
\qquad
\operatorname{XYZ}(i)=
\begin{cases}
X, & CV_i\leq CV_X,\\
Y, & CV_X<CV_i\leq CV_Y,\\
Z, & CV_i>CV_Y.
\end{cases}
\]

The implementation uses the chromosome fields `v_a_threshold`, `v_b_threshold`, `cv_x_threshold`, and `cv_y_threshold` to apply these rules. The intersection of the two classifications gives one of nine categories, such as AX or CZ.

### Performance proxies and fitness

For chromosome \(c\), let \(S(c)\), \(C(c)\), and \(L(c)\) denote the stability, cost-reduction, and service-improvement proxies calculated by `calculate_performance_metrics`. Its scalar fitness is

\[
F(c)=w_sS(c)+w_cC(c)+w_lL(c),
\qquad
w_s,w_c,w_l\geq 0,
\qquad
w_s+w_c+w_l=1.
\]

The weights are part of the chromosome, so the search considers both threshold choices and the relative importance assigned to the three proxies. This is a weighted-sum objective, not a Pareto-front calculation. Each proxy is bounded in the implementation; because they have different interpretations and ranges, their weighted sum is a demonstration objective rather than a calibrated economic measure.

For stability, the code compares CV values within each category containing at least two SKUs. Its category score is \(\max(0,1-s_{CV}/(\bar{CV}+0.01))\), and the overall score is the unweighted mean across eligible categories. The cost proxy compares category-factor-weighted annual value with a baseline equal to 80% of total annual value. The service proxy compares the SKU-count-weighted category service factors with a baseline of 0.85. These definitions explain exactly what the reported scores represent; neither proxy is a direct measurement of realized cost or service.

### Search logic

For a candidate with fitness \(F\), simulated annealing proposes a mutated neighbor with fitness \(F'\). Improving candidates are accepted. A worsening candidate is accepted with probability

\[
P(\text{accept})=\exp\left(\frac{F'-F}{T}\right),
\]

where \(T>0\) is the current temperature. Temperature is multiplied by the cooling rate after each proposal. The genetic algorithm supplies broad population-based exploration; annealing supplies local refinement. The implementation returns the best candidate encountered during each annealing run, so an accepted temporary decline does not replace that run's best result.

If \(P\) is the population size, \(G\) the number of generations, \(N\) the number of SKUs, and \(I\) the number of annealing proposals, evaluating the population is approximately \(O(PN)\) per generation. Pairwise population-diversity tracking adds \(O(P^2)\) per generation, and annealing adds approximately \(O(IN)\) at each refinement. Thus the overall work scales roughly as \(O(G(PN+P^2)+IN)\), apart from constants and the number of annealing calls.

### What to examine

- Whether the best and mean fitness histories change over generations.
- How population diversity changes as candidates reproduce and mutate.
- The nine-cell ABC/XYZ distribution produced by the returned thresholds.
- How the three proxy metrics contribute to the scalar fitness.
- Whether the chosen thresholds and weights are plausible for the explicitly stated proxy model.

The notebook does not claim a particular optimized threshold, cost reduction, service improvement, or classification stability in advance. Those values depend on executing the code and should be interpreted only in light of the model assumptions above.

In [ ]:
from dataclasses import dataclass
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A fixed seed makes the stochastic search reproducible in a compatible environment.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported successfully for the hybrid GA-SA search.")

In [ ]:
@dataclass
class SKU:
    """A stock-keeping unit with annual value and monthly demand observations."""

    sku_id: str
    annual_value: float
    monthly_demands: List[float]

    def __post_init__(self) -> None:
        if not self.monthly_demands:
            raise ValueError("Each SKU must have at least one demand observation.")
        if self.annual_value < 0:
            raise ValueError("Annual value must be nonnegative.")

        self.avg_demand = float(np.mean(self.monthly_demands))
        self.std_demand = float(np.std(self.monthly_demands, ddof=1)) if len(self.monthly_demands) > 1 else 0.0
        self.cv = self.std_demand / self.avg_demand if self.avg_demand > 0 else float("inf")


@dataclass
class Chromosome:
    """Genetic representation of classification thresholds and objective weights."""

    v_a_threshold: float
    v_b_threshold: float
    cv_x_threshold: float
    cv_y_threshold: float
    w_stability: float
    w_cost: float
    w_service: float
    fitness: float = 0.0


@dataclass
class OptimizationResult:
    """Best solution and search histories produced by the hybrid algorithm."""

    best_chromosome: Chromosome
    fitness_history: List[float]
    avg_fitness_history: List[float]
    diversity_history: List[float]
    temperature_history: List[float]
    optimization_time: float
    generations_completed: int


@dataclass
class PerformanceMetrics:
    """Proxy measures used to evaluate a threshold configuration."""

    classification_stability: float
    inventory_cost_reduction: float
    service_level_improvement: float
    total_fitness: float

print("Data classes are ready for the hybrid GA-SA search.")

In [ ]:
def initialize_population(
    population_size: int,
    value_range: Tuple[float, float],
    cv_range: Tuple[float, float],
) -> List[Chromosome]:
    """Create a population of ordered thresholds and normalized objective weights."""
    if population_size < 3:
        raise ValueError("Population size must be at least 3 for tournament selection.")
    if value_range[0] > value_range[1] or cv_range[0] > cv_range[1]:
        raise ValueError("Each data range must be ordered from minimum to maximum.")

    population = []

    for _ in range(population_size):
        # Separate sampling bands encourage distinct high- and lower-value cutoffs.
        v_a = np.random.uniform(value_range[0] * 0.7, value_range[1] * 0.9)
        v_b = np.random.uniform(value_range[0] * 0.1, value_range[1] * 0.3)
        if v_b >= v_a:
            v_a, v_b = max(v_a, v_b), min(v_a, v_b)

        # The X cutoff must be below the Y cutoff to form ordered variability classes.
        cv_x = np.random.uniform(cv_range[0], cv_range[1] * 0.4)
        cv_y = np.random.uniform(cv_range[1] * 0.3, cv_range[1] * 0.7)
        if cv_y <= cv_x:
            cv_x, cv_y = min(cv_x, cv_y), max(cv_x, cv_y)

        w_stability = np.random.uniform(0.2, 0.6)
        w_cost = np.random.uniform(0.2, 0.5)
        w_service = max(0.1, 1.0 - w_stability - w_cost)
        total_weight = w_stability + w_cost + w_service
        w_stability /= total_weight
        w_cost /= total_weight
        w_service /= total_weight

        population.append(
            Chromosome(
                v_a_threshold=v_a,
                v_b_threshold=v_b,
                cv_x_threshold=cv_x,
                cv_y_threshold=cv_y,
                w_stability=w_stability,
                w_cost=w_cost,
                w_service=w_service,
            )
        )

    return population


def classify_skus_with_thresholds(
    skus: List[SKU], chromosome: Chromosome
) -> Dict[str, List[SKU]]:
    """Assign every SKU to exactly one of the nine ABC/XYZ categories."""
    if chromosome.v_a_threshold < chromosome.v_b_threshold:
        raise ValueError("The A-class value threshold must be at least the B-class threshold.")
    if chromosome.cv_x_threshold > chromosome.cv_y_threshold:
        raise ValueError("The X-class CV threshold must not exceed the Y-class threshold.")

    categories = {
        f"{abc}{xyz}": []
        for abc in ("A", "B", "C")
        for xyz in ("X", "Y", "Z")
    }

    for sku in skus:
        if sku.annual_value >= chromosome.v_a_threshold:
            abc_class = "A"
        elif sku.annual_value >= chromosome.v_b_threshold:
            abc_class = "B"
        else:
            abc_class = "C"

        if sku.cv <= chromosome.cv_x_threshold:
            xyz_class = "X"
        elif sku.cv <= chromosome.cv_y_threshold:
            xyz_class = "Y"
        else:
            xyz_class = "Z"

        categories[abc_class + xyz_class].append(sku)

    return categories

print("Population initialization and ABC/XYZ classification are ready.")

In [ ]:
def calculate_performance_metrics(
    skus: List[SKU],
    categories: Dict[str, List[SKU]],
    chromosome: Chromosome,
) -> PerformanceMetrics:
    """Calculate the stated stability, cost, and service proxies and their weighted fitness."""
    if not skus:
        raise ValueError("At least one SKU is required to calculate performance metrics.")

    # This is within-category similarity in CV, not stability over time.
    stability_scores = []
    for skus_in_category in categories.values():
        if len(skus_in_category) >= 2:
            cvs = np.array([sku.cv for sku in skus_in_category], dtype=float)
            cv_std = np.std(cvs)
            cv_mean = np.mean(cvs)
            stability_scores.append(max(0.0, 1.0 - cv_std / (cv_mean + 0.01)))

    classification_stability = (
        float(np.mean(stability_scores)) if stability_scores else 0.5
    )

    # Category factors define an illustrative proxy, not measured inventory expenditure.
    cost_factors = {
        "AX": 0.8, "AY": 0.85, "AZ": 0.9,
        "BX": 0.7, "BY": 0.75, "BZ": 0.8,
        "CX": 0.6, "CY": 0.65, "CZ": 0.7,
    }
    total_value = sum(sku.annual_value for sku in skus)
    optimized_cost = sum(
        sum(sku.annual_value for sku in categories[category]) * cost_factors[category]
        for category in categories
    )
    baseline_cost = total_value * 0.8
    if baseline_cost > 0:
        inventory_cost_reduction = (baseline_cost - optimized_cost) / baseline_cost
    else:
        inventory_cost_reduction = 0.0
    inventory_cost_reduction = float(np.clip(inventory_cost_reduction, -0.5, 0.5))

    # Category service factors define a count-weighted proxy, not an observed fill rate.
    service_alignment = {
        "AX": 0.95, "AY": 0.90, "AZ": 0.85,
        "BX": 0.90, "BY": 0.85, "BZ": 0.80,
        "CX": 0.85, "CY": 0.80, "CZ": 0.75,
    }
    weighted_service = sum(
        len(categories[category]) * service_alignment[category]
        for category in categories
    ) / len(skus)
    baseline_service = 0.85
    service_level_improvement = float(
        np.clip((weighted_service - baseline_service) / baseline_service, -0.3, 0.3)
    )

    weights = np.array(
        [chromosome.w_stability, chromosome.w_cost, chromosome.w_service],
        dtype=float,
    )
    if np.any(weights < 0) or not np.isclose(weights.sum(), 1.0, rtol=1e-9, atol=1e-9):
        raise ValueError("Objective weights must be nonnegative and sum to 1.")

    total_fitness = float(
        chromosome.w_stability * classification_stability
        + chromosome.w_cost * inventory_cost_reduction
        + chromosome.w_service * service_level_improvement
    )

    return PerformanceMetrics(
        classification_stability=classification_stability,
        inventory_cost_reduction=inventory_cost_reduction,
        service_level_improvement=service_level_improvement,
        total_fitness=total_fitness,
    )


def evaluate_population(
    population: List[Chromosome], skus: List[SKU]
) -> List[PerformanceMetrics]:
    """Evaluate each chromosome and store its scalar fitness for selection."""
    metrics_list = []
    for chromosome in population:
        categories = classify_skus_with_thresholds(skus, chromosome)
        metrics = calculate_performance_metrics(skus, categories, chromosome)
        chromosome.fitness = metrics.total_fitness
        metrics_list.append(metrics)
    return metrics_list

print("Proxy performance and weighted fitness functions are ready.")

In [ ]:
def tournament_selection(
    population: List[Chromosome], tournament_size: int = 3
) -> Chromosome:
    """Select the fittest member of a randomly sampled tournament."""
    if tournament_size < 1 or tournament_size > len(population):
        raise ValueError("Tournament size must be between 1 and the population size.")
    tournament_indices = np.random.choice(len(population), tournament_size, replace=False)
    tournament = [population[index] for index in tournament_indices]
    return max(tournament, key=lambda chromosome: chromosome.fitness)


def crossover(
    parent1: Chromosome, parent2: Chromosome
) -> Tuple[Chromosome, Chromosome]:
    """Apply uniform crossover, choosing each gene from either parent."""
    genes1 = [
        parent1.v_a_threshold, parent1.v_b_threshold,
        parent1.cv_x_threshold, parent1.cv_y_threshold,
        parent1.w_stability, parent1.w_cost, parent1.w_service,
    ]
    genes2 = [
        parent2.v_a_threshold, parent2.v_b_threshold,
        parent2.cv_x_threshold, parent2.cv_y_threshold,
        parent2.w_stability, parent2.w_cost, parent2.w_service,
    ]
    child1_genes = []
    child2_genes = []

    for gene1, gene2 in zip(genes1, genes2):
        if np.random.random() < 0.5:
            child1_genes.append(gene1)
            child2_genes.append(gene2)
        else:
            child1_genes.append(gene2)
            child2_genes.append(gene1)

    child1 = Chromosome(*child1_genes)
    child2 = Chromosome(*child2_genes)
    return child1, child2


def mutate(
    chromosome: Chromosome,
    mutation_rate: float = 0.1,
    value_range: Optional[Tuple[float, float]] = None,
    cv_range: Optional[Tuple[float, float]] = None,
) -> Chromosome:
    """Apply Gaussian gene mutations and restore threshold and weight feasibility."""
    if not 0.0 <= mutation_rate <= 1.0:
        raise ValueError("Mutation rate must be between 0 and 1.")

    mutated = Chromosome(
        v_a_threshold=chromosome.v_a_threshold,
        v_b_threshold=chromosome.v_b_threshold,
        cv_x_threshold=chromosome.cv_x_threshold,
        cv_y_threshold=chromosome.cv_y_threshold,
        w_stability=chromosome.w_stability,
        w_cost=chromosome.w_cost,
        w_service=chromosome.w_service,
    )

    if value_range is not None:
        if np.random.random() < mutation_rate:
            mutated.v_a_threshold *= np.random.normal(1.0, 0.1)
        if np.random.random() < mutation_rate:
            mutated.v_b_threshold *= np.random.normal(1.0, 0.1)
        mutated.v_a_threshold = float(np.clip(mutated.v_a_threshold, *value_range))
        mutated.v_b_threshold = float(np.clip(mutated.v_b_threshold, *value_range))

    # Sorting preserves both cutoffs while ensuring the ABC rule remains ordered.
    if mutated.v_b_threshold > mutated.v_a_threshold:
        mutated.v_a_threshold, mutated.v_b_threshold = (
            mutated.v_b_threshold,
            mutated.v_a_threshold,
        )

    if cv_range is not None:
        if np.random.random() < mutation_rate:
            mutated.cv_x_threshold *= np.random.normal(1.0, 0.05)
        if np.random.random() < mutation_rate:
            mutated.cv_y_threshold *= np.random.normal(1.0, 0.05)
        mutated.cv_x_threshold = float(np.clip(mutated.cv_x_threshold, *cv_range))
        mutated.cv_y_threshold = float(np.clip(mutated.cv_y_threshold, *cv_range))

    # Ordered CV cutoffs are required for the X, Y, and Z intervals.
    if mutated.cv_y_threshold < mutated.cv_x_threshold:
        mutated.cv_x_threshold, mutated.cv_y_threshold = (
            mutated.cv_y_threshold,
            mutated.cv_x_threshold,
        )

    if np.random.random() < mutation_rate:
        mutated.w_stability *= np.random.normal(1.0, 0.05)
    if np.random.random() < mutation_rate:
        mutated.w_cost *= np.random.normal(1.0, 0.05)
    if np.random.random() < mutation_rate:
        mutated.w_service *= np.random.normal(1.0, 0.05)

    weights = np.maximum(
        [mutated.w_stability, mutated.w_cost, mutated.w_service], 1e-12
    )
    weights = weights / weights.sum()
    mutated.w_stability, mutated.w_cost, mutated.w_service = weights.tolist()
    return mutated

print("Tournament selection, uniform crossover, and Gaussian mutation are ready.")

In [ ]:
def simulated_annealing_refinement(
    best_chromosome: Chromosome,
    skus: List[SKU],
    initial_temp: float = 100.0,
    cooling_rate: float = 0.95,
    min_temp: float = 1.0,
    max_iterations: int = 100,
) -> Chromosome:
    """Refine a candidate with temperature-based acceptance of neighboring solutions."""
    if not skus:
        raise ValueError("At least one SKU is required for annealing.")
    if initial_temp <= 0 or min_temp < 0 or not 0 < cooling_rate < 1:
        raise ValueError("Temperature must be positive and cooling rate must be in (0, 1).")
    if max_iterations < 0:
        raise ValueError("Maximum iterations must be nonnegative.")

    current = Chromosome(
        best_chromosome.v_a_threshold,
        best_chromosome.v_b_threshold,
        best_chromosome.cv_x_threshold,
        best_chromosome.cv_y_threshold,
        best_chromosome.w_stability,
        best_chromosome.w_cost,
        best_chromosome.w_service,
        best_chromosome.fitness,
    )
    current.fitness = calculate_performance_metrics(
        skus, classify_skus_with_thresholds(skus, current), current
    ).total_fitness
    best_seen = current
    temperature = initial_temp

    values = [sku.annual_value for sku in skus]
    cvs = [sku.cv for sku in skus]
    value_range = (min(values), max(values))
    finite_cvs = [cv for cv in cvs if np.isfinite(cv)]
    cv_range = (min(finite_cvs), max(finite_cvs)) if finite_cvs else (0.0, 0.0)

    for _ in range(max_iterations):
        if temperature < min_temp:
            break

        neighbor = mutate(
            current,
            mutation_rate=0.3,
            value_range=value_range,
            cv_range=cv_range,
        )
        neighbor.fitness = calculate_performance_metrics(
            skus,
            classify_skus_with_thresholds(skus, neighbor),
            neighbor,
        ).total_fitness

        delta_fitness = neighbor.fitness - current.fitness
        if delta_fitness > 0:
            current = neighbor
        else:
            acceptance_probability = np.exp(delta_fitness / temperature)
            if np.random.random() < acceptance_probability:
                current = neighbor

        if current.fitness > best_seen.fitness:
            best_seen = current

        temperature *= cooling_rate

    return best_seen


def calculate_population_diversity(population: List[Chromosome]) -> float:
    """Return the mean pairwise Euclidean distance between chromosome gene vectors."""
    if len(population) < 2:
        return 0.0

    gene_vectors = [
        np.array(
            [
                chromosome.v_a_threshold,
                chromosome.v_b_threshold,
                chromosome.cv_x_threshold,
                chromosome.cv_y_threshold,
                chromosome.w_stability,
                chromosome.w_cost,
                chromosome.w_service,
            ],
            dtype=float,
        )
        for chromosome in population
    ]

    total_distance = 0.0
    comparisons = 0
    for i in range(len(gene_vectors)):
        for j in range(i + 1, len(gene_vectors)):
            total_distance += float(np.linalg.norm(gene_vectors[i] - gene_vectors[j]))
            comparisons += 1

    return total_distance / comparisons

print("Simulated annealing and population-diversity functions are ready.")

In [ ]:
def hybrid_ga_sa_optimization(
    skus: List[SKU],
    population_size: int = 50,
    generations: int = 50,
    crossover_rate: float = 0.8,
    mutation_rate: float = 0.1,
) -> OptimizationResult:
    """Run genetic evolution with periodic and final simulated-annealing refinement."""
    if not skus:
        raise ValueError("At least one SKU is required for optimization.")
    if generations < 1:
        raise ValueError("At least one generation is required.")
    if not 0.0 <= crossover_rate <= 1.0:
        raise ValueError("Crossover rate must be between 0 and 1.")
    if population_size < 3:
        raise ValueError("Population size must be at least 3 for tournament selection.")

    start_time = time.time()
    values = [sku.annual_value for sku in skus]
    cvs = [sku.cv for sku in skus]
    value_range = (min(values), max(values))
    finite_cvs = [cv for cv in cvs if np.isfinite(cv)]
    cv_range = (min(finite_cvs), max(finite_cvs)) if finite_cvs else (0.0, 0.0)

    population = initialize_population(population_size, value_range, cv_range)
    evaluate_population(population, skus)

    fitness_history = []
    avg_fitness_history = []
    diversity_history = []
    temperature_history = []

    for generation in range(generations):
        fitness_values = [chromosome.fitness for chromosome in population]
        fitness_history.append(max(fitness_values))
        avg_fitness_history.append(float(np.mean(fitness_values)))
        diversity_history.append(calculate_population_diversity(population))

        new_population = []
        elite = max(population, key=lambda chromosome: chromosome.fitness)
        new_population.append(
            Chromosome(
                elite.v_a_threshold,
                elite.v_b_threshold,
                elite.cv_x_threshold,
                elite.cv_y_threshold,
                elite.w_stability,
                elite.w_cost,
                elite.w_service,
                elite.fitness,
            )
        )

        while len(new_population) < population_size:
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)

            if np.random.random() < crossover_rate:
                child1, child2 = crossover(parent1, parent2)
            else:
                child1, child2 = crossover(parent1, parent2)
                child1, child2 = (
                    Chromosome(
                        parent1.v_a_threshold, parent1.v_b_threshold,
                        parent1.cv_x_threshold, parent1.cv_y_threshold,
                        parent1.w_stability, parent1.w_cost, parent1.w_service,
                    ),
                    Chromosome(
                        parent2.v_a_threshold, parent2.v_b_threshold,
                        parent2.cv_x_threshold, parent2.cv_y_threshold,
                        parent2.w_stability, parent2.w_cost, parent2.w_service,
                    ),
                )

            child1 = mutate(child1, mutation_rate, value_range, cv_range)
            child2 = mutate(child2, mutation_rate, value_range, cv_range)
            new_population.extend([child1, child2])

        population = new_population[:population_size]
        evaluate_population(population, skus)

        if generation % 10 == 0 and generation > 0:
            current_best = max(population, key=lambda chromosome: chromosome.fitness)
            refined = simulated_annealing_refinement(current_best, skus)
            worst_index = min(
                range(len(population)), key=lambda index: population[index].fitness
            )
            if refined.fitness > population[worst_index].fitness:
                population[worst_index] = refined
            # Each annealing call restarts at its configured initial temperature.
            temperature_history.append(100.0)

    final_best = max(population, key=lambda chromosome: chromosome.fitness)
    final_best = simulated_annealing_refinement(final_best, skus)
    optimization_time = time.time() - start_time

    return OptimizationResult(
        best_chromosome=final_best,
        fitness_history=fitness_history,
        avg_fitness_history=avg_fitness_history,
        diversity_history=diversity_history,
        temperature_history=temperature_history,
        optimization_time=optimization_time,
        generations_completed=generations,
    )

print("The hybrid GA-SA optimization procedure is ready.")

In [ ]:
# Generate the example dataset with the stated fixed random seed.
print("Creating the SKU dataset for hybrid GA-SA optimization...")
np.random.seed(42)
test_skus = []

for i in range(1000):
    category = np.random.choice(
        [
            "high_stable", "high_moderate", "high_volatile",
            "medium_stable", "medium_moderate", "medium_volatile",
            "low_stable", "low_moderate", "low_volatile",
        ],
        p=[0.08, 0.06, 0.04, 0.12, 0.10, 0.08, 0.20, 0.18, 0.14],
    )

    if category.startswith("high"):
        annual_value = np.random.uniform(80000, 500000)
        base_demand = np.random.uniform(50, 150)
    elif category.startswith("medium"):
        annual_value = np.random.uniform(20000, 80000)
        base_demand = np.random.uniform(20, 80)
    else:
        annual_value = np.random.uniform(5000, 25000)
        base_demand = np.random.uniform(5, 30)

    if "stable" in category:
        cv_target = np.random.uniform(0.02, 0.08)
    elif "moderate" in category:
        cv_target = np.random.uniform(0.08, 0.25)
    else:
        cv_target = np.random.uniform(0.25, 0.80)

    std_demand = base_demand * cv_target
    monthly_demands = np.random.normal(base_demand, std_demand, 12).clip(0, None).tolist()
    test_skus.append(SKU(f"SKU_{i + 1000:04d}", annual_value, monthly_demands))

assert len(test_skus) == 1000
print(f"Created {len(test_skus)} SKUs.")
print(
    "Annual value range: "
    f"${min(sku.annual_value for sku in test_skus):,.2f} to "
    f"${max(sku.annual_value for sku in test_skus):,.2f}"
)
print(
    "CV range: "
    f"{min(sku.cv for sku in test_skus):.3f} to "
    f"{max(sku.cv for sku in test_skus):.3f}"
)

high_value = [sku for sku in test_skus if sku.annual_value > 80000]
medium_value = [sku for sku in test_skus if 20000 <= sku.annual_value <= 80000]
low_value = [sku for sku in test_skus if sku.annual_value < 20000]
print("\nValue distribution:")
print(f"  High value (> $80,000): {len(high_value)} SKUs ({len(high_value) / len(test_skus):.1%})")
print(f"  Medium value ($20,000-$80,000): {len(medium_value)} SKUs ({len(medium_value) / len(test_skus):.1%})")
print(f"  Low value (< $20,000): {len(low_value)} SKUs ({len(low_value) / len(test_skus):.1%})")

stable_demand = [sku for sku in test_skus if sku.cv < 0.10]
moderate_demand = [sku for sku in test_skus if 0.10 <= sku.cv < 0.25]
volatile_demand = [sku for sku in test_skus if sku.cv >= 0.25]
print("\nDemand-variability distribution:")
print(f"  Stable (CV < 10%): {len(stable_demand)} SKUs ({len(stable_demand) / len(test_skus):.1%})")
print(f"  Moderate (10% <= CV < 25%): {len(moderate_demand)} SKUs ({len(moderate_demand) / len(test_skus):.1%})")
print(f"  Volatile (CV >= 25%): {len(volatile_demand)} SKUs ({len(volatile_demand) / len(test_skus):.1%})")

In [ ]:
print("Running the hybrid genetic algorithm with simulated annealing...")
print("=" * 70)

POPULATION_SIZE = 50
GENERATIONS = 50
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.1

print("Search parameters:")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Generations: {GENERATIONS}")
print(f"  Crossover rate: {CROSSOVER_RATE}")
print(f"  Mutation rate: {MUTATION_RATE}")
print(f"  Dataset size: {len(test_skus)} SKUs")

optimization_result = hybrid_ga_sa_optimization(
    skus=test_skus,
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
)
best = optimization_result.best_chromosome
final_categories = classify_skus_with_thresholds(test_skus, best)
final_metrics = calculate_performance_metrics(test_skus, final_categories, best)

assert sum(len(category_skus) for category_skus in final_categories.values()) == len(test_skus)
assert np.isclose(best.fitness, final_metrics.total_fitness, rtol=1e-9, atol=1e-9)
assert np.isclose(
    best.w_stability + best.w_cost + best.w_service,
    1.0,
    rtol=1e-9,
    atol=1e-9,
)

print("\nBest threshold configuration:")
print(f"  A-class annual-value threshold: ${best.v_a_threshold:,.2f}")
print(f"  B-class annual-value threshold: ${best.v_b_threshold:,.2f}")
print(f"  X-class CV threshold: {best.cv_x_threshold:.3f}")
print(f"  Y-class CV threshold: {best.cv_y_threshold:.3f}")
print("\nObjective weights:")
print(f"  Stability proxy weight: {best.w_stability:.3f}")
print(f"  Cost proxy weight: {best.w_cost:.3f}")
print(f"  Service proxy weight: {best.w_service:.3f}")

print("\nProxy performance metrics:")
print(f"  Within-category CV stability proxy: {final_metrics.classification_stability:.3f}")
print(f"  Inventory cost-reduction proxy: {final_metrics.inventory_cost_reduction:.3f}")
print(f"  Service-level improvement proxy: {final_metrics.service_level_improvement:.3f}")
print(f"  Weighted fitness: {final_metrics.total_fitness:.3f}")

print("\nABC/XYZ distribution:")
for category in sorted(final_categories):
    count = len(final_categories[category])
    if count:
        print(f"  {category}: {count} SKUs ({count / len(test_skus):.1%})")

print("\nSearch summary:")
print(f"  Generations completed: {optimization_result.generations_completed}")
print(f"  Best fitness recorded at the start of the final generation: {optimization_result.fitness_history[-1]:.4f}")
print(f"  Best fitness after final annealing: {best.fitness:.4f}")
print(f"  Initial-to-final-generation best-fitness change: {optimization_result.fitness_history[-1] - optimization_result.fitness_history[0]:.4f}")
print(f"  Average-fitness change: {optimization_result.avg_fitness_history[-1] - optimization_result.avg_fitness_history[0]:.4f}")
print(f"  Elapsed optimization time: {optimization_result.optimization_time:.2f} seconds")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Hybrid GA-SA Search Analysis", fontsize=16, fontweight="bold")

# The histories describe the population before reproduction in each generation.
generations = np.arange(len(optimization_result.fitness_history))
ax1 = axes[0, 0]
ax1.plot(generations, optimization_result.fitness_history, linewidth=2, label="Best fitness")
ax1.plot(generations, optimization_result.avg_fitness_history, linestyle="--", linewidth=2, label="Mean fitness")
ax1.set_xlabel("Generation")
ax1.set_ylabel("Weighted fitness (unitless)")
ax1.set_title("Fitness during Genetic Evolution")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(generations, optimization_result.diversity_history, color="tab:red", linewidth=2)
ax2.set_xlabel("Generation")
ax2.set_ylabel("Mean pairwise gene distance")
ax2.set_title("Population Diversity")
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
if optimization_result.temperature_history:
    refinement_numbers = np.arange(1, len(optimization_result.temperature_history) + 1)
    ax3.plot(
        refinement_numbers,
        optimization_result.temperature_history,
        color="tab:orange",
        marker="s",
        linewidth=2,
    )
    ax3.set_xlabel("Periodic annealing call")
    ax3.set_ylabel("Initial temperature")
    ax3.set_title("Annealing Restart Temperature")
    ax3.grid(True, alpha=0.3)
else:
    ax3.text(0.5, 0.5, "No periodic annealing call", ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Annealing Restart Temperature")
    ax3.set_axis_off()

ax4 = axes[1, 0]
matrix_data = np.zeros((3, 3), dtype=int)
abc_index = {"A": 0, "B": 1, "C": 2}
xyz_index = {"X": 0, "Y": 1, "Z": 2}
for category, category_skus in final_categories.items():
    matrix_data[abc_index[category[0]], xyz_index[category[1]]] = len(category_skus)

image = ax4.imshow(matrix_data, cmap="Blues", alpha=0.8)
fig.colorbar(image, ax=ax4, label="Number of SKUs")
ax4.set_xticks([0, 1, 2], labels=["X (lower CV)", "Y (middle CV)", "Z (higher CV)"])
ax4.set_yticks([0, 1, 2], labels=["A (higher value)", "B (middle value)", "C (lower value)"])
ax4.set_xlabel("Demand-variability class")
ax4.set_ylabel("Annual-value class")
ax4.set_title("Optimized ABC/XYZ Classification")
for row in range(3):
    for column in range(3):
        count = int(matrix_data[row, column])
        ax4.text(column, row, f"{count}\n({count / len(test_skus):.1%})", ha="center", va="center", color="black")

ax5 = axes[1, 1]
metric_names = ["Stability\nproxy", "Cost reduction\nproxy", "Service improvement\nproxy"]
metric_values = [
    final_metrics.classification_stability,
    final_metrics.inventory_cost_reduction,
    final_metrics.service_level_improvement,
]
bar_colors = ["tab:green" if value >= 0 else "tab:red" for value in metric_values]
bars = ax5.bar(metric_names, metric_values, color=bar_colors, alpha=0.75)
ax5.set_ylabel("Proxy score (unitless)")
ax5.set_title("Components of the Weighted Objective")
low = min(0.0, min(metric_values))
high = max(0.0, max(metric_values))
margin = max(0.03, (high - low) * 0.15)
ax5.set_ylim(low - margin, high + margin)
ax5.axhline(0, color="black", linewidth=0.8)
ax5.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, metric_values):
    offset = margin * 0.12 if value >= 0 else -margin * 0.12
    ax5.text(bar.get_x() + bar.get_width() / 2, value + offset, f"{value:.3f}", ha="center", va="bottom" if value >= 0 else "top")

ax6 = axes[1, 2]
value_data = [sku.annual_value for sku in test_skus]
baseline_value_thresholds = [
    float(np.percentile(value_data, 80)),
    float(np.percentile(value_data, 20)),
]
threshold_positions = np.arange(4)
width = 0.35
ax6.bar(
    threshold_positions[:2] - width / 2,
    baseline_value_thresholds,
    width,
    label="Percentile reference: 80th / 20th",
    color="lightblue",
)
ax6.bar(
    threshold_positions[:2] + width / 2,
    [best.v_a_threshold, best.v_b_threshold],
    width,
    label="GA-SA value thresholds",
    color="tab:orange",
)
ax6.set_xticks(threshold_positions, labels=["A value", "B value", "X CV", "Y CV"])
ax6.set_xlabel("Threshold type")
ax6.set_ylabel("Annual value")
ax6.set_title("Thresholds and Reference Values")
ax6.grid(True, axis="y", alpha=0.3)

ax6_cv = ax6.twinx()
ax6_cv.bar(
    threshold_positions[2:] - width / 2,
    [0.10, 0.25],
    width,
    label="CV reference: 0.10 / 0.25",
    color="lightgreen",
)
ax6_cv.bar(
    threshold_positions[2:] + width / 2,
    [best.cv_x_threshold, best.cv_y_threshold],
    width,
    label="GA-SA CV thresholds",
    color="tab:purple",
    alpha=0.75,
)
ax6_cv.set_ylabel("Coefficient of variation (unitless)")
handles1, labels1 = ax6.get_legend_handles_labels()
handles2, labels2 = ax6_cv.get_legend_handles_labels()
ax6.legend(handles1 + handles2, labels1 + labels2, fontsize=8, loc="upper right")

fig.tight_layout()
plt.show()
print("Search histories, classification, proxy metrics, and thresholds are displayed.")

In [ ]:
print("SOLUTION CONSISTENCY CHECKS")
print("=" * 70)

# Recompute all reported quantities from the returned chromosome and input data.
verified_categories = classify_skus_with_thresholds(test_skus, best)
verified_metrics = calculate_performance_metrics(test_skus, verified_categories, best)
classified_count = sum(len(category_skus) for category_skus in verified_categories.values())
assert classified_count == len(test_skus)
assert best.v_a_threshold >= best.v_b_threshold
assert best.cv_x_threshold <= best.cv_y_threshold
assert np.isclose(best.fitness, verified_metrics.total_fitness, rtol=1e-9, atol=1e-9)
assert np.isclose(
    best.w_stability + best.w_cost + best.w_service,
    1.0,
    rtol=1e-9,
    atol=1e-9,
)

print(f"All SKUs classified exactly once: {classified_count} of {len(test_skus)}")
print(f"ABC threshold order is feasible: {best.v_a_threshold >= best.v_b_threshold}")
print(f"XYZ threshold order is feasible: {best.cv_x_threshold <= best.cv_y_threshold}")
print(f"Objective weights sum to: {best.w_stability + best.w_cost + best.w_service:.12f}")
print(f"Stored and recomputed fitness agree: {np.isclose(best.fitness, verified_metrics.total_fitness, rtol=1e-9, atol=1e-9)}")
print("\nInterpretation:")
print("These checks verify feasibility and internal consistency for the stated proxy model.")
print("They do not establish global optimality or validate the proxies against operational outcomes.")

## Summary and interpretation

This notebook applies a hybrid genetic algorithm and simulated annealing to search for ABC/XYZ thresholds and weights for a weighted proxy-fitness objective. The genetic algorithm explores a population through tournament selection, uniform crossover, mutation, and elitism. Periodic and final annealing runs refine candidates using temperature-based acceptance of neighboring solutions.

The reported threshold configuration is the best candidate encountered by this stochastic search, not a certified global optimum. Fitness histories describe the population at the start of each generation; the final annealing step can further improve the returned candidate. Population diversity is the mean pairwise Euclidean distance across genes, whose different scales mean it is a descriptive indicator rather than a normalized diversity measure.

The three metric values must be read according to their definitions: the stability measure is within-category similarity in CV, the cost measure is a category-factor proxy relative to a fixed value-based baseline, and the service measure is a count-weighted category proxy relative to a fixed baseline. They are not measured classification stability, realized cost savings, or observed service-level changes. The scalar fitness combines these proxies using the chromosome's normalized weights.

A practical application would require business-validated cost and service measures, a carefully justified stability definition, and operational constraints that reflect the decision context. Those additions would change the model inputs or objective and are outside this example's scope.